<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/05_GES_Aware_Genomic_RAG_Cell_7A3_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# GES-RAG Experiment 2 — Cell 7A3

**Run only after the exact Cell 7A2 terminal PASS.**

This notebook materializes and checksum-freezes the authorized T1 scores:

- Frozen Full-GES P(stable) and instability risk
- Frozen No-star-GES P(stable) and instability risk
- Frozen Stage 6A combined-metadata instability risk and its six audit components

It first reproduces the frozen T0 Stage 4C probabilities exactly, then scores all 100,920 T1 rows without fitting or tuning.

### Not authorized here

- Threshold selection, rank construction, or evidence exclusion
- Outcome loading or temporal-performance analysis
- Evidence packets, RAG corpus, embeddings, retrieval, questions, prompts, or LLM calls

Use **Runtime → Run all**. Stop after the terminal Cell 7A3 PASS. A later RAG cell requires a separate frozen authorization.

> **Corrected V2:** This version fixes one fail-closed QC false positive. The audit flag `score_rank_constructed=False` was being mistaken for a materialized rank column. No feature reconstruction, model scoring, policy calculation, or frozen scientific boundary was changed.


In [1]:

# Environment check only — this cell does not install or modify packages.
import sys
import numpy as np
import pandas as pd
import pyarrow
import sklearn
import joblib
import scipy

print("Python       :", sys.version.split()[0])
print("NumPy        :", np.__version__)
print("pandas       :", pd.__version__)
print("PyArrow      :", pyarrow.__version__)
print("scikit-learn :", sklearn.__version__)
print("joblib       :", joblib.__version__)
print("SciPy        :", scipy.__version__)


Python       : 3.12.13
NumPy        : 2.0.2
pandas       : 2.2.2
PyArrow      : 18.1.0
scikit-learn : 1.6.1
joblib       : 1.5.3
SciPy        : 1.16.3


In [2]:

# ==================================================================================================
# EXPERIMENT 2 — STAGE 7A — CELL 7A3
# FROZEN T1 FULL-GES, NO-STAR-GES, AND COMBINED-METADATA SCORE MATERIALIZATION
#
# PURPOSE
#   1. Verify the exact Cell 7A2 terminal PASS and every frozen upstream checksum.
#   2. Independently reconstruct the frozen T1 model features.
#   3. Verify T0 predict_proba reproduction against the frozen Stage 4C score table.
#   4. Apply the already-fitted Full-GES and No-star-GES pipelines to T1 without fitting.
#   5. Apply the exact frozen Stage 6A combined-metadata policy to T1.
#   6. Materialize one row-level T1 score table and freeze its QC/manifest package.
#
# STRICT BOUNDARY
#   - No model fitting, refitting, recalibration, or tuning
#   - No thresholding, rank construction, hard exclusion, or weight optimization
#   - No outcome loading or temporal-performance calculation
#   - No evidence-packet or RAG-corpus construction
#   - No embeddings, retrieval, prompts, question set, or LLM calls
#
# A terminal PASS freezes Cell 7A3 only. It does not automatically authorize a later RAG cell.
# ==================================================================================================

from collections import OrderedDict, Counter
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import math
import os
import platform
import re
import sys
import time

import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import sparse
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# --------------------------------------------------------------------------------------------------
# 1. GOOGLE DRIVE AND PROJECT PATHS
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive")

if not DRIVE_ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")

if not DRIVE_ROOT.exists():
    raise FileNotFoundError("Google Drive is not mounted at /content/drive/MyDrive.")

ROOT = DRIVE_ROOT / "GES_RAG_Temporal_Study"
NOTEBOOK_NAME = "05_GES_Aware_Genomic_RAG_Cell_7A3_V2.ipynb"
CELL_ID = "7A3"
PACKAGE_VERSION = "v1"

STAGE4_DATA_DIR = ROOT / "data_processed" / "stage4_ges"
STAGE4_MODEL_DIR = ROOT / "models" / "stage4_ges"
STAGE4_CONFIG_DIR = ROOT / "configs" / "stage4_ges"
STAGE6_CONFIG_DIR = ROOT / "configs" / "stage6_temporal_validation"
STAGE7_CONFIG_DIR = ROOT / "configs" / "stage7_rag"
STAGE7_DATA_DIR = ROOT / "data_processed" / "stage7_rag"
STAGE7_TABLE_DIR = ROOT / "outputs" / "tables" / "stage7_rag"
STAGE7_QC_DIR = ROOT / "outputs" / "quality_checks" / "stage7_rag"

for directory in [STAGE7_CONFIG_DIR, STAGE7_DATA_DIR, STAGE7_TABLE_DIR, STAGE7_QC_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------------------------------------------------------
# 2. EXACT FROZEN INPUT PATHS
# --------------------------------------------------------------------------------------------------

CELL_7A1_MANIFEST = (
    STAGE7_CONFIG_DIR / "cell_7a1_t1_corpus_source_preflight_manifest_v1.json"
)


CELL_7A2_MANIFEST = (
    STAGE7_CONFIG_DIR
    / "cell_7a2_feature_reconstruction_model_applicability_manifest_v1.json"
)

T1_PARQUET = ROOT / "data_interim" / "t1_rcv_target_genes_harmonized_v1.parquet"
T1_FREEZE_MANIFEST = (
    ROOT / "configs" / "clinvar_t1_target_gene_rcv_extraction_validation_manifest_v1.json"
)

STAGE4A_FEATURE_TABLE = (
    STAGE4_DATA_DIR / "stage4a_t0_ges_baseline_features_v1.parquet"
)
STAGE4A_FEATURE_SPEC = (
    STAGE4_CONFIG_DIR / "stage4a_t0_feature_specification_v1.json"
)
STAGE4A_QC = (
    STAGE4_CONFIG_DIR / "stage4a_t0_feature_qc_report_v1.json"
)
STAGE4A_MANIFEST = (
    STAGE4_CONFIG_DIR / "stage4a_t0_feature_freeze_manifest_v1.json"
)

STAGE4B_TABLE = (
    STAGE4_DATA_DIR / "stage4b_t0_feature_transforms_and_weak_labels_v1.parquet"
)
STAGE4B_TRANSFORM_PARAMETERS = (
    STAGE4_CONFIG_DIR / "stage4b_t0_feature_transform_parameters_v1.json"
)
STAGE4B_WEAK_LABEL_RULES = (
    STAGE4_CONFIG_DIR / "stage4b_weak_label_rules_v1.json"
)
STAGE4B_QC = (
    STAGE4_CONFIG_DIR / "stage4b_weak_label_qc_report_v1.json"
)
STAGE4B_MANIFEST = (
    STAGE4_CONFIG_DIR / "stage4b_weak_label_freeze_manifest_v1.json"
)

FULL_MODEL_PATH = STAGE4_MODEL_DIR / "stage4c_full_ges_logistic_model_v1.joblib"
NO_STAR_MODEL_PATH = STAGE4_MODEL_DIR / "stage4c_no_star_ges_logistic_model_v1.joblib"
STAGE4C_SCORE_TABLE = STAGE4_DATA_DIR / "stage4c_t0_full_and_no_star_ges_scores_v1.parquet"
STAGE4C_MODEL_SPEC = STAGE4_CONFIG_DIR / "stage4c_ges_model_specification_v1.json"
STAGE4C_QC = STAGE4_CONFIG_DIR / "stage4c_ges_model_qc_report_v1.json"
STAGE4C_MANIFEST = STAGE4_CONFIG_DIR / "stage4c_ges_model_freeze_manifest_v1.json"

STAGE6A_POLICY = STAGE6_CONFIG_DIR / "stage6a_comparator_score_policy_v1.json"


# --------------------------------------------------------------------------------------------------
# 3. EXPECTED HASHES, COUNTS, FEATURES, AND MODEL SETTINGS
# --------------------------------------------------------------------------------------------------

EXPECTED_HASHES = OrderedDict([
    ("cell_7a1_manifest", "84e509d97f01fb8dc0b6ad0c7e24de762e8923b9068fc835bf328105f79e946d"),
    ("t1_parquet", "5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c"),
    ("t1_freeze_manifest", "7eaeff0fee3df96973130f721d6c1f2a02fd9108e85af7b2743cdd7750a4372e"),
    ("stage4a_feature_table", "c100b3781e6801425f622f5d091376abfe0939e48c0a792af32f6eebe6401f16"),
    ("stage4a_feature_spec", "fc00146efe5da9b3fbe740bb42ca99d157252cdefc045650f9e88d54d8fcfa8b"),
    ("stage4a_qc", "4bf72af66aa800fa9f12fdc8598bd06ebc859bb31c97ad97cf7fc11ef1614cf1"),
    ("stage4a_manifest", "2b844ef2dbc0c3e5e57493886a7533587350e1b4b4c76fc9d445ecda8a528fa0"),
    ("stage4b_table", "c2e9e4f96cd1f61d3962e88d28557dfa1221729b8f79f8fd4d2dafb34847bdb8"),
    ("stage4b_transform_parameters", "baeab167e19381138f93c51ba2fa00e4eeed684b36122c80cd2bf9fc4fe4b08d"),
    ("stage4b_weak_label_rules", "3d78e66cea1fed5c75ef1cab1b7cf44d3d3d7bfae50909173bae6c3a0e0bff61"),
    ("stage4b_qc", "03b14efb6d297fd517043c8ff952977715415723f191d079d968c111368d31a9"),
    ("stage4b_manifest", "e766061442e6d4610f661f44a619e41b45dc6363b28f6176d3f9a71f8215c63f"),
    ("stage4c_full_model", "0b4a87b16f768484cbdae168fc226cec5e978521172fb03510bfb1ea3e78fa30"),
    ("stage4c_no_star_model", "6c3fe4fc7fe8fdde7b8f0f0d608c48e66a07945effb8c67c6b98d35e1955257c"),
    ("stage4c_score_table", "d871ee9087f83be2b0ee954d283aa92212a639a35e3ea26d5cf042e83019f5ac"),
    ("stage4c_model_spec", "d754c715c990f42cecd64259ca2c420427b9602c669dc7be9e80dee9554f61f6"),
    ("stage4c_qc", "3a1a90d3a8946fd35bde52324a625077f3127ac57f231b20d206d70b2c678ec7"),
    ("stage4c_manifest", "c0d8008a4db80c67f5b1c568ddba3496b2411b29bce0db1eb20e63f26613d4ee"),
    ("stage6a_policy", "dd7e95dc785e77b04c289ef50817b1ddac7516a436d83b5734dbf3cd35b1248d"),
])

ARTIFACT_PATHS = OrderedDict([
    ("cell_7a1_manifest", CELL_7A1_MANIFEST),
    ("t1_parquet", T1_PARQUET),
    ("t1_freeze_manifest", T1_FREEZE_MANIFEST),
    ("stage4a_feature_table", STAGE4A_FEATURE_TABLE),
    ("stage4a_feature_spec", STAGE4A_FEATURE_SPEC),
    ("stage4a_qc", STAGE4A_QC),
    ("stage4a_manifest", STAGE4A_MANIFEST),
    ("stage4b_table", STAGE4B_TABLE),
    ("stage4b_transform_parameters", STAGE4B_TRANSFORM_PARAMETERS),
    ("stage4b_weak_label_rules", STAGE4B_WEAK_LABEL_RULES),
    ("stage4b_qc", STAGE4B_QC),
    ("stage4b_manifest", STAGE4B_MANIFEST),
    ("stage4c_full_model", FULL_MODEL_PATH),
    ("stage4c_no_star_model", NO_STAR_MODEL_PATH),
    ("stage4c_score_table", STAGE4C_SCORE_TABLE),
    ("stage4c_model_spec", STAGE4C_MODEL_SPEC),
    ("stage4c_qc", STAGE4C_QC),
    ("stage4c_manifest", STAGE4C_MANIFEST),
    ("stage6a_policy", STAGE6A_POLICY),
])

EXPECTED_T1_ROWS = 100_920
EXPECTED_T1_COLUMNS = 36
EXPECTED_STAGE4A_ROWS = 71_659
EXPECTED_STAGE4A_COLUMNS = 30
EXPECTED_STAGE4B_ROWS = 71_659
EXPECTED_STAGE4B_COLUMNS = 43

EXPECTED_T1_GENE_COUNTS = {
    "BRCA1": 32_603,
    "BRCA2": 49_221,
    "MLH1": 13_684,
    "EGFR": 5_412,
}

EXPECTED_T1_AXIS_COUNTS = {
    "GermlineClassification": 97_526,
    "OncogenicityClassification": 52,
    "SomaticClinicalImpact": 25,
    "NoClassification": 3_317,
}

EXPECTED_CELL_7A1_DECISION = (
    "PASS_STAGE7A1_T1_CORPUS_SOURCE_VERIFIED_CHECKSUM_PROTECTED_"
    "TOP_LEVEL_AND_NESTED_SCHEMA_INVENTORIED_EVIDENCE_PACKET_"
    "DERIVABILITY_AUDITED_STAGE7A2_PREFLIGHT_ONLY"
)


EXPECTED_CELL_7A2_MANIFEST_SHA256 = (
    "9c1b93467778e4ef580474dc85940a7dc2e5b840e0d46f889e2d3e19cbe46c34"
)
EXPECTED_CELL_7A2_DECISION = (
    "PASS_STAGE7A2_FROZEN_FEATURE_TRANSFORMS_AND_MODEL_PACKAGES_VERIFIED_"
    "T1_FEATURE_RECONSTRUCTION_AND_PREPROCESSING_APPLICABILITY_CONFIRMED_"
    "NO_SCORING_STAGE7A3_FROZEN_T1_SCORE_MATERIALIZATION_AUTHORIZED"
)

T1_CUTOFF = pd.Timestamp("2025-12-27")

FULL_FEATURES = [
    "recency_score",
    "recency_missing_flag",
    "submitter_diversity_score",
    "review_confidence",
    "aggregate_conflict_flag",
    "scv_group_entropy_normalized",
]

NO_STAR_FEATURES = [
    "recency_score",
    "recency_missing_flag",
    "submitter_diversity_score",
    "aggregate_conflict_flag",
    "scv_group_entropy_normalized",
]

EXPECTED_MODEL_SETTINGS = {
    "penalty": "l2",
    "C": 1.0,
    "solver": "lbfgs",
    "class_weight": None,
    "max_iter": 2_000,
    "tol": 1e-6,
    "fit_intercept": True,
    "random_state": 42,
}



# --------------------------------------------------------------------------------------------------
# 4. OUTPUT PATHS
# --------------------------------------------------------------------------------------------------

OUTPUTS = OrderedDict([
    (
        "score_table",
        STAGE7_DATA_DIR / "cell_7a3_t1_frozen_ges_and_metadata_scores_v1.parquet",
    ),
    (
        "schema_inventory",
        STAGE7_TABLE_DIR / "cell_7a3_t1_score_schema_inventory_v1.csv",
    ),
    (
        "score_qc_inventory",
        STAGE7_TABLE_DIR / "cell_7a3_t1_score_range_qc_inventory_v1.csv",
    ),
    (
        "materialization_report",
        STAGE7_QC_DIR / "cell_7a3_t1_score_materialization_report_v1.json",
    ),
    (
        "qc",
        STAGE7_QC_DIR / "cell_7a3_t1_score_materialization_qc_v1.json",
    ),
    (
        "manifest",
        STAGE7_CONFIG_DIR / "cell_7a3_t1_score_materialization_manifest_v1.json",
    ),
])


# --------------------------------------------------------------------------------------------------
# 5. GENERAL HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    path = Path(path)
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()


def sidecar_path(path: Path) -> Path:
    return Path(str(path) + ".sha256")


def read_sidecar_hash(path: Path) -> str:
    text = Path(path).read_text(encoding="utf-8").strip()
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", text)
    if not matches:
        raise ValueError(f"No SHA-256 value found in sidecar: {path}")
    return matches[0].lower()


def sidecar_is_valid(path: Path) -> bool:
    path = Path(path)
    sidecar = sidecar_path(path)
    return (
        path.exists()
        and sidecar.exists()
        and read_sidecar_hash(sidecar) == sha256_file(path)
    )


def verify_exact_hash(label: str, path: Path, expected: str) -> str:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required frozen artifact for {label}:\n{path}"
        )
    observed = sha256_file(path)
    if observed != expected:
        raise AssertionError(
            f"SHA-256 mismatch for {label}.\n"
            f"Expected: {expected}\nObserved: {observed}\nPath: {path}"
        )
    return observed


def json_native(value):
    if isinstance(value, dict):
        return {str(key): json_native(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_native(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()
    if isinstance(value, np.ndarray):
        return [json_native(item) for item in value.tolist()]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if value is pd.NA:
        return None
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


def stable_write_bytes(path: Path, payload: bytes) -> str:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(
        f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}"
    )
    temporary.write_bytes(payload)
    proposed_hash = sha256_file(temporary)

    if path.exists():
        existing_hash = sha256_file(path)
        if existing_hash != proposed_hash:
            temporary.unlink(missing_ok=True)
            raise RuntimeError(
                "Refusing to overwrite a nonidentical frozen Cell 7A3 artifact.\n"
                f"Path: {path}\n"
                f"Existing SHA-256: {existing_hash}\n"
                f"Proposed SHA-256: {proposed_hash}"
            )
        temporary.unlink(missing_ok=True)
    else:
        os.replace(temporary, path)

    return sha256_file(path)


def stable_write_json(path: Path, payload: dict) -> str:
    data = (
        json.dumps(
            json_native(payload),
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n"
    ).encode("utf-8")
    return stable_write_bytes(path, data)


def stable_write_csv(path: Path, frame: pd.DataFrame) -> str:
    data = frame.to_csv(
        index=False,
        lineterminator="\n",
        float_format="%.12g",
    ).encode("utf-8")
    return stable_write_bytes(path, data)


def write_sidecar(path: Path) -> str:
    path = Path(path)
    payload = f"{sha256_file(path)}  {path.name}\n".encode("utf-8")
    stable_write_bytes(sidecar_path(path), payload)
    return sha256_file(sidecar_path(path))


def resolve_column(columns, aliases, label, required=True):
    exact = {str(column).lower(): str(column) for column in columns}
    for alias in aliases:
        if alias.lower() in exact:
            return exact[alias.lower()]
    if required:
        raise KeyError(
            f"Could not resolve required column '{label}'.\n"
            f"Aliases checked: {aliases}\n"
            f"Available columns: {list(columns)}"
        )
    return None


def parse_json_value(value):
    if value is None or value is pd.NA:
        return None, "missing"
    try:
        if pd.isna(value):
            return None, "missing"
    except Exception:
        pass

    if isinstance(value, (dict, list)):
        return value, "native"

    text = str(value).strip()
    if not text:
        return None, "blank"

    try:
        return json.loads(text), "parsed"
    except json.JSONDecodeError:
        return None, "parse_error"


def normalize_gene(value) -> str:
    allowed = {"BRCA1", "BRCA2", "MLH1", "EGFR"}
    parsed, status = parse_json_value(value)

    if status == "parse_error":
        parsed = [str(value).strip()]
    if isinstance(parsed, str):
        parsed = [parsed]
    if not isinstance(parsed, list):
        return ""

    genes = sorted({
        str(gene).strip().upper()
        for gene in parsed
        if str(gene).strip().upper() in allowed
    })
    return genes[0] if len(genes) == 1 else ""


def boolish_to_float(value):
    if value is None or value is pd.NA:
        return np.nan
    try:
        if pd.isna(value):
            return np.nan
    except Exception:
        pass

    if isinstance(value, (bool, np.bool_)):
        return float(value)
    if isinstance(value, (int, float, np.integer, np.floating)):
        number = float(value)
        return number if number in {0.0, 1.0} else np.nan

    text = str(value).strip().lower()
    if text in {"true", "t", "yes", "y", "1"}:
        return 1.0
    if text in {"false", "f", "no", "n", "0"}:
        return 0.0
    return np.nan


def numeric_dict_sum(mapping):
    if not isinstance(mapping, dict):
        return np.nan, 1

    total = 0.0
    invalid = 0
    for value in mapping.values():
        try:
            number = float(value)
        except (TypeError, ValueError):
            invalid += 1
            continue
        if not np.isfinite(number) or number < 0:
            invalid += 1
            continue
        total += number
    return total, invalid


def normalized_entropy_from_group_counts(value):
    parsed, status = parse_json_value(value)

    if status == "parse_error":
        return np.nan, np.nan, 0, "parse_error"
    if parsed is None:
        return np.nan, np.nan, 0, status

    total, invalid = numeric_dict_sum(parsed)
    if invalid > 0:
        return np.nan, total, 0, "invalid_numeric_value"

    positive_counts = np.asarray(
        [float(item) for item in parsed.values() if float(item) > 0],
        dtype=float,
    )

    if len(positive_counts) == 0:
        return np.nan, total, 0, "no_positive_counts"
    if len(positive_counts) == 1:
        return 0.0, total, 1, "single_group"

    probabilities = positive_counts / positive_counts.sum()
    entropy = -float(np.sum(probabilities * np.log(probabilities)))
    normalized = float(
        np.clip(entropy / math.log(len(positive_counts)), 0.0, 1.0)
    )
    return normalized, total, int(len(positive_counts)), "derived"


def extract_pipeline(artifact, label):
    if isinstance(artifact, Pipeline):
        return artifact, "<top-level>"

    preferred_keys = [
        "pipeline",
        "model_pipeline",
        "fitted_pipeline",
        "sklearn_pipeline",
        "model",
        "estimator",
        "classifier",
    ]

    if isinstance(artifact, dict):
        for key in preferred_keys:
            if key in artifact and isinstance(artifact[key], Pipeline):
                return artifact[key], f"[{key!r}]"

    matches = []
    visited = set()

    def walk(obj, location="<top-level>", depth=0):
        if depth > 8 or id(obj) in visited:
            return
        visited.add(id(obj))

        if isinstance(obj, Pipeline):
            matches.append((location, obj))
            return

        if isinstance(obj, dict):
            for key, item in obj.items():
                walk(item, f"{location}[{key!r}]", depth + 1)
        elif isinstance(obj, (list, tuple)):
            for index, item in enumerate(obj):
                walk(item, f"{location}[{index}]", depth + 1)
        else:
            for attribute in preferred_keys:
                if hasattr(obj, attribute):
                    try:
                        walk(
                            getattr(obj, attribute),
                            f"{location}.{attribute}",
                            depth + 1,
                        )
                    except Exception:
                        pass

    walk(artifact)

    unique = {}
    for location, pipeline in matches:
        unique.setdefault(id(pipeline), (location, pipeline))
    found = list(unique.values())

    if len(found) != 1:
        raise TypeError(
            f"Expected exactly one sklearn Pipeline in {label}; "
            f"found {len(found)}."
        )
    return found[0][1], found[0][0]


def get_exact_component(pipeline, component_type):
    matches = [
        (name, step)
        for name, step in pipeline.steps
        if isinstance(step, component_type)
    ]
    if len(matches) != 1:
        raise AssertionError(
            f"Expected exactly one {component_type.__name__}; "
            f"found {len(matches)}."
        )
    return matches[0]


def transformed_values_are_finite(values) -> bool:
    if sparse.issparse(values):
        return bool(np.isfinite(values.data).all())
    return bool(np.isfinite(np.asarray(values)).all())


def model_settings_match(classifier: LogisticRegression) -> bool:
    return bool(
        classifier.penalty == EXPECTED_MODEL_SETTINGS["penalty"]
        and np.isclose(float(classifier.C), EXPECTED_MODEL_SETTINGS["C"])
        and classifier.solver == EXPECTED_MODEL_SETTINGS["solver"]
        and classifier.class_weight == EXPECTED_MODEL_SETTINGS["class_weight"]
        and int(classifier.max_iter) == EXPECTED_MODEL_SETTINGS["max_iter"]
        and np.isclose(float(classifier.tol), EXPECTED_MODEL_SETTINGS["tol"])
        and bool(classifier.fit_intercept)
            == EXPECTED_MODEL_SETTINGS["fit_intercept"]
        and classifier.random_state == EXPECTED_MODEL_SETTINGS["random_state"]
    )


def find_feature_order(artifact, pipeline, expected_features):
    expected_features = [str(item) for item in expected_features]
    candidate_keys = {
        "feature_columns",
        "feature_names",
        "features",
        "input_features",
        "model_features",
        "predictor_columns",
    }
    candidates = []

    if hasattr(pipeline, "feature_names_in_"):
        candidates.append(
            ("pipeline.feature_names_in_", list(pipeline.feature_names_in_))
        )

    visited = set()

    def walk(obj, location="<top-level>", depth=0):
        if depth > 7 or id(obj) in visited:
            return
        visited.add(id(obj))

        if isinstance(obj, dict):
            for key, value in obj.items():
                key_text = str(key).lower()
                if key_text in candidate_keys and isinstance(
                    value, (list, tuple, np.ndarray, pd.Index)
                ):
                    candidates.append(
                        (f"{location}[{key!r}]", [str(item) for item in value])
                    )
                walk(value, f"{location}[{key!r}]", depth + 1)
        elif isinstance(obj, (list, tuple)):
            for index, value in enumerate(obj):
                walk(value, f"{location}[{index}]", depth + 1)

    walk(artifact)

    exact_matches = [
        (location, values)
        for location, values in candidates
        if values == expected_features
    ]

    if exact_matches:
        return exact_matches[0][1], exact_matches[0][0], True

    if candidates:
        location, values = candidates[0]
        return values, location, False

    return [], "<not-found>", False


def preprocess_without_scoring(pipeline, frame):
    classifier_positions = [
        index
        for index, (_, step) in enumerate(pipeline.steps)
        if isinstance(step, LogisticRegression)
    ]
    if len(classifier_positions) != 1:
        raise AssertionError(
            "Expected exactly one LogisticRegression position in the pipeline."
        )

    classifier_index = classifier_positions[0]
    transformed = frame.copy()

    for step_name, step in pipeline.steps[:classifier_index]:
        if not hasattr(step, "transform"):
            raise AssertionError(
                f"Pre-classifier step '{step_name}' has no transform() method."
            )
        transformed = step.transform(transformed)

    return transformed



# --------------------------------------------------------------------------------------------------
# 6. VERIFY ALL FROZEN INPUT ARTIFACTS AND CELL 7A2 AUTHORIZATION
# --------------------------------------------------------------------------------------------------

observed_hashes = OrderedDict()
for key, path in ARTIFACT_PATHS.items():
    observed_hashes[key] = verify_exact_hash(key, path, EXPECTED_HASHES[key])

# Preserve ancestry verification through Cell 7A1.
if not sidecar_is_valid(CELL_7A1_MANIFEST):
    raise AssertionError("Cell 7A1 manifest sidecar verification failed.")
cell_7a1_payload = json.loads(CELL_7A1_MANIFEST.read_text(encoding="utf-8"))
if cell_7a1_payload.get("terminal_decision") != EXPECTED_CELL_7A1_DECISION:
    raise AssertionError("Cell 7A1 terminal decision is not the frozen expected value.")

# Cell 7A3 is authorized only by the exact frozen Cell 7A2 terminal PASS.
verify_exact_hash(
    "cell_7a2_manifest",
    CELL_7A2_MANIFEST,
    EXPECTED_CELL_7A2_MANIFEST_SHA256,
)
if not sidecar_is_valid(CELL_7A2_MANIFEST):
    raise AssertionError("Cell 7A2 manifest sidecar verification failed.")

cell_7a2_payload = json.loads(CELL_7A2_MANIFEST.read_text(encoding="utf-8"))
if cell_7a2_payload.get("terminal_decision") != EXPECTED_CELL_7A2_DECISION:
    raise AssertionError("Cell 7A2 terminal decision does not authorize Cell 7A3.")
if cell_7a2_payload.get("next_authorized_cell", {}).get("cell_id") != "7A3":
    raise AssertionError("Cell 7A2 manifest does not identify Cell 7A3 as next authorized.")
qc_summary_7a2 = cell_7a2_payload.get("qc", {})
if not (
    int(qc_summary_7a2.get("passed_checks", -1)) == 62
    and int(qc_summary_7a2.get("failed_checks", -1)) == 0
    and int(qc_summary_7a2.get("total_checks", -1)) == 62
):
    raise AssertionError("Cell 7A2 manifest does not record the expected 62/62 PASS.")

# Reverify every frozen artifact referenced by the Cell 7A2 manifest.
cell_7a2_output_paths = OrderedDict()
for output_key, record in cell_7a2_payload.get("output_artifacts", {}).items():
    path = Path(record["path"])
    expected_hash = str(record["sha256"])
    verify_exact_hash(f"cell_7a2_output:{output_key}", path, expected_hash)
    if not sidecar_is_valid(path):
        raise AssertionError(f"Cell 7A2 output sidecar failed: {path}")
    cell_7a2_output_paths[f"cell_7a2_output:{output_key}"] = path

upstream_paths = OrderedDict(ARTIFACT_PATHS)
upstream_paths["cell_7a2_manifest"] = CELL_7A2_MANIFEST
upstream_paths.update(cell_7a2_output_paths)

immutable_hashes_before = {
    key: sha256_file(path)
    for key, path in upstream_paths.items()
}


# --------------------------------------------------------------------------------------------------
# 7. LOAD EXACT FROZEN TRANSFORMATION PARAMETERS
# --------------------------------------------------------------------------------------------------

stage4b_transform_payload = json.loads(
    STAGE4B_TRANSFORM_PARAMETERS.read_text(encoding="utf-8")
)

frozen_recency_parameters = stage4b_transform_payload.get(
    "recency_transformation", {}
)
frozen_submitter_parameters = stage4b_transform_payload.get(
    "submitter_diversity_transformation", {}
)

required_frozen_parameter_keys = {
    "maximum_observation_window_days":
        frozen_recency_parameters.get("maximum_observation_window_days"),
    "minimum_log_count":
        frozen_submitter_parameters.get("minimum_log_count"),
    "maximum_log_count":
        frozen_submitter_parameters.get("maximum_log_count"),
}

missing_frozen_parameter_keys = [
    key
    for key, value in required_frozen_parameter_keys.items()
    if value is None
]

if missing_frozen_parameter_keys:
    raise KeyError(
        "The frozen Stage 4B transformation-parameter JSON is missing: "
        f"{missing_frozen_parameter_keys}"
    )

MAXIMUM_OBSERVATION_WINDOW_DAYS = float(
    required_frozen_parameter_keys["maximum_observation_window_days"]
)
SUBMITTER_LOG_MIN = float(
    required_frozen_parameter_keys["minimum_log_count"]
)
SUBMITTER_LOG_MAX = float(
    required_frozen_parameter_keys["maximum_log_count"]
)

if not (
    np.isfinite(MAXIMUM_OBSERVATION_WINDOW_DAYS)
    and MAXIMUM_OBSERVATION_WINDOW_DAYS > 0
    and np.isfinite(SUBMITTER_LOG_MIN)
    and np.isfinite(SUBMITTER_LOG_MAX)
    and SUBMITTER_LOG_MAX > SUBMITTER_LOG_MIN
):
    raise ValueError(
        "Frozen Stage 4B transformation parameters are not numerically valid."
    )


# --------------------------------------------------------------------------------------------------
# 8. VERIFY PARQUET DIMENSIONS AND BUILD FROZEN ARTIFACT INVENTORY
# --------------------------------------------------------------------------------------------------

t1_meta = pq.ParquetFile(T1_PARQUET).metadata
stage4a_meta = pq.ParquetFile(STAGE4A_FEATURE_TABLE).metadata
stage4b_meta = pq.ParquetFile(STAGE4B_TABLE).metadata

artifact_inventory_rows = []

for key, path in ARTIFACT_PATHS.items():
    row = {
        "artifact_key": key,
        "path": str(path),
        "file_name": path.name,
        "suffix": path.suffix.lower(),
        "expected_sha256": EXPECTED_HASHES[key],
        "observed_sha256": observed_hashes[key],
        "hash_verified": observed_hashes[key] == EXPECTED_HASHES[key],
        "bytes": int(path.stat().st_size),
        "sidecar_present": sidecar_path(path).exists(),
        "sidecar_verified": (
            sidecar_is_valid(path) if sidecar_path(path).exists() else False
        ),
        "rows": None,
        "columns": None,
    }

    if path.suffix.lower() == ".parquet":
        metadata = pq.ParquetFile(path).metadata
        row["rows"] = int(metadata.num_rows)
        row["columns"] = int(metadata.num_columns)

    artifact_inventory_rows.append(row)

artifact_inventory = pd.DataFrame(artifact_inventory_rows)


# --------------------------------------------------------------------------------------------------
# 9. REPRODUCE THE FROZEN STAGE 4B TRANSFORMATIONS ON T0
# --------------------------------------------------------------------------------------------------

stage4a_columns = pq.ParquetFile(
    STAGE4A_FEATURE_TABLE
).schema_arrow.names
stage4b_columns = pq.ParquetFile(
    STAGE4B_TABLE
).schema_arrow.names

stage4a_required_columns = [
    "t0_row_order",
    "rcv_accession",
    "recency_days",
    "recency_missing_flag",
    "unique_submitter_count",
    "log1p_unique_submitter_count",
    "aggregate_review_stars",
]

stage4b_required_columns = list(dict.fromkeys([
    "t0_row_order",
    "rcv_accession",
    "recency_score",
    "recency_missing_flag",
    "submitter_diversity_score",
    "review_confidence",
] + FULL_FEATURES))

missing_stage4a_columns = [
    column
    for column in stage4a_required_columns
    if column not in stage4a_columns
]
missing_stage4b_columns = [
    column
    for column in stage4b_required_columns
    if column not in stage4b_columns
]

if missing_stage4a_columns:
    raise KeyError(
        "Frozen Stage 4A table is missing required raw-source columns: "
        f"{missing_stage4a_columns}"
    )

if missing_stage4b_columns:
    raise KeyError(
        "Frozen Stage 4B table is missing required transformed columns: "
        f"{missing_stage4b_columns}"
    )

t0_raw = pd.read_parquet(
    STAGE4A_FEATURE_TABLE,
    columns=stage4a_required_columns,
).copy()

t0 = pd.read_parquet(
    STAGE4B_TABLE,
    columns=stage4b_required_columns,
).copy()

if len(t0_raw) != EXPECTED_STAGE4A_ROWS:
    raise RuntimeError(
        f"Loaded Stage 4A row count is {len(t0_raw):,}; "
        f"expected {EXPECTED_STAGE4A_ROWS:,}."
    )

if len(t0) != EXPECTED_STAGE4B_ROWS:
    raise RuntimeError(
        f"Loaded Stage 4B row count is {len(t0):,}; "
        f"expected {EXPECTED_STAGE4B_ROWS:,}."
    )

t0_raw_row_order = pd.to_numeric(
    t0_raw["t0_row_order"], errors="raise"
).astype("int64")
t0_row_order = pd.to_numeric(
    t0["t0_row_order"], errors="raise"
).astype("int64")

t0_raw_rcv = (
    t0_raw["rcv_accession"].astype("string").str.strip().str.upper()
)
t0_rcv = (
    t0["rcv_accession"].astype("string").str.strip().str.upper()
)

stage4a_row_order_valid = bool(
    t0_raw_row_order.nunique(dropna=False) == EXPECTED_STAGE4A_ROWS
    and np.array_equal(
        t0_raw_row_order.to_numpy(),
        np.arange(EXPECTED_STAGE4A_ROWS, dtype=np.int64),
    )
)

stage4b_row_order_valid = bool(
    t0_row_order.nunique(dropna=False) == EXPECTED_STAGE4B_ROWS
    and np.array_equal(
        t0_row_order.to_numpy(),
        np.arange(EXPECTED_STAGE4B_ROWS, dtype=np.int64),
    )
)

stage4a_stage4b_row_order_alignment_verified = bool(
    np.array_equal(
        t0_raw_row_order.to_numpy(),
        t0_row_order.to_numpy(),
    )
)

stage4a_stage4b_rcv_alignment_verified = bool(
    t0_raw_rcv.notna().all()
    and t0_rcv.notna().all()
    and not t0_raw_rcv.fillna("").eq("").any()
    and not t0_rcv.fillna("").eq("").any()
    and t0_raw_rcv.nunique(dropna=False) == EXPECTED_STAGE4A_ROWS
    and t0_rcv.nunique(dropna=False) == EXPECTED_STAGE4B_ROWS
    and np.array_equal(
        t0_raw_rcv.to_numpy(dtype=str),
        t0_rcv.to_numpy(dtype=str),
    )
)

if not stage4a_row_order_valid:
    raise RuntimeError("Frozen Stage 4A row-order verification failed.")
if not stage4b_row_order_valid:
    raise RuntimeError("Frozen Stage 4B row-order verification failed.")
if not stage4a_stage4b_row_order_alignment_verified:
    raise RuntimeError("Stage 4A and Stage 4B row-order alignment failed.")
if not stage4a_stage4b_rcv_alignment_verified:
    raise RuntimeError("Stage 4A and Stage 4B RCV-key alignment failed.")

# Recency formula
recency_days_t0 = pd.to_numeric(
    t0_raw["recency_days"], errors="coerce"
).astype(float)
saved_recency_t0 = pd.to_numeric(
    t0["recency_score"], errors="coerce"
).astype(float)

expected_recency_t0 = (
    1.0 - recency_days_t0 / MAXIMUM_OBSERVATION_WINDOW_DAYS
).clip(0.0, 1.0)
expected_recency_t0.loc[recency_days_t0.isna()] = np.nan

recency_formula_mismatches = int((~np.isclose(
    saved_recency_t0.to_numpy(),
    expected_recency_t0.to_numpy(),
    rtol=1e-12,
    atol=1e-12,
    equal_nan=True,
)).sum())

# Missingness formula
saved_missing_stage4a_t0 = pd.to_numeric(
    t0_raw["recency_missing_flag"], errors="coerce"
).astype(float)
saved_missing_stage4b_t0 = pd.to_numeric(
    t0["recency_missing_flag"], errors="coerce"
).astype(float)
expected_missing_t0 = recency_days_t0.isna().astype(float)

recency_missing_stage4a_mismatches = int((~np.isclose(
    saved_missing_stage4a_t0.to_numpy(),
    expected_missing_t0.to_numpy(),
    rtol=0.0,
    atol=0.0,
    equal_nan=True,
)).sum())

recency_missing_stage4b_mismatches = int((~np.isclose(
    saved_missing_stage4b_t0.to_numpy(),
    expected_missing_t0.to_numpy(),
    rtol=0.0,
    atol=0.0,
    equal_nan=True,
)).sum())

recency_missing_mismatches = (
    recency_missing_stage4a_mismatches
    + recency_missing_stage4b_mismatches
)

# Submitter formula
submitter_count_t0 = pd.to_numeric(
    t0_raw["unique_submitter_count"], errors="raise"
).astype(float)
expected_log_submitter_t0 = np.log1p(submitter_count_t0)
saved_log_submitter_t0 = pd.to_numeric(
    t0_raw["log1p_unique_submitter_count"], errors="raise"
).astype(float)

log_submitter_mismatches = int((~np.isclose(
    saved_log_submitter_t0.to_numpy(),
    expected_log_submitter_t0.to_numpy(),
    rtol=1e-12,
    atol=1e-12,
)).sum())

observed_log_min = float(expected_log_submitter_t0.min())
observed_log_max = float(expected_log_submitter_t0.max())

expected_submitter_score_t0 = (
    (expected_log_submitter_t0 - SUBMITTER_LOG_MIN)
    / (SUBMITTER_LOG_MAX - SUBMITTER_LOG_MIN)
).clip(0.0, 1.0)

saved_submitter_score_t0 = pd.to_numeric(
    t0["submitter_diversity_score"], errors="raise"
).astype(float)

submitter_formula_mismatches = int((~np.isclose(
    saved_submitter_score_t0.to_numpy(),
    expected_submitter_score_t0.to_numpy(),
    rtol=1e-12,
    atol=1e-12,
)).sum())

# Review-confidence formula
expected_review_t0 = pd.to_numeric(
    t0_raw["aggregate_review_stars"], errors="raise"
).astype(float)
saved_review_t0 = pd.to_numeric(
    t0["review_confidence"], errors="raise"
).astype(float)

review_formula_mismatches = int((~np.isclose(
    saved_review_t0.to_numpy(),
    expected_review_t0.to_numpy(),
    rtol=0.0,
    atol=0.0,
)).sum())

formula_audit = pd.DataFrame([
    {
        "transformation": "recency_score",
        "rows_audited": len(t0),
        "mismatch_count": recency_formula_mismatches,
        "exactly_reproduced": recency_formula_mismatches == 0,
    },
    {
        "transformation": "recency_missing_flag_stage4a",
        "rows_audited": len(t0_raw),
        "mismatch_count": recency_missing_stage4a_mismatches,
        "exactly_reproduced": recency_missing_stage4a_mismatches == 0,
    },
    {
        "transformation": "recency_missing_flag_stage4b",
        "rows_audited": len(t0),
        "mismatch_count": recency_missing_stage4b_mismatches,
        "exactly_reproduced": recency_missing_stage4b_mismatches == 0,
    },
    {
        "transformation": "log1p_unique_submitter_count",
        "rows_audited": len(t0_raw),
        "mismatch_count": log_submitter_mismatches,
        "exactly_reproduced": log_submitter_mismatches == 0,
    },
    {
        "transformation": "submitter_diversity_score",
        "rows_audited": len(t0),
        "mismatch_count": submitter_formula_mismatches,
        "exactly_reproduced": submitter_formula_mismatches == 0,
    },
    {
        "transformation": "review_confidence",
        "rows_audited": len(t0),
        "mismatch_count": review_formula_mismatches,
        "exactly_reproduced": review_formula_mismatches == 0,
    },
])


# --------------------------------------------------------------------------------------------------
# 10. LOAD T1 SOURCE AND RECONSTRUCT THE SIX FEATURES IN MEMORY
# --------------------------------------------------------------------------------------------------

t1_schema_columns = pq.ParquetFile(T1_PARQUET).schema_arrow.names

t1_columns = OrderedDict([
    (
        "rcv_accession",
        resolve_column(
            t1_schema_columns,
            ["rcv_accession", "t1_rcv_accession"],
            "T1 RCV accession",
        ),
    ),
    (
        "target_gene",
        resolve_column(
            t1_schema_columns,
            ["target_genes_json", "target_gene", "gene"],
            "T1 target gene",
        ),
    ),
    (
        "classification_axis",
        resolve_column(
            t1_schema_columns,
            ["aggregate_classification_axis", "classification_axis"],
            "T1 classification axis",
        ),
    ),
    (
        "embedded_cutoff",
        resolve_column(
            t1_schema_columns,
            [
                "embedded_data_cutoff_date",
                "data_cutoff_date",
                "embedded_cutoff_date",
            ],
            "T1 embedded cutoff date",
        ),
    ),
    (
        "aggregate_last_evaluated",
        resolve_column(
            t1_schema_columns,
            ["aggregate_last_evaluated", "last_evaluated"],
            "T1 aggregate last evaluated",
        ),
    ),
    (
        "unique_submitter_count",
        resolve_column(
            t1_schema_columns,
            ["unique_submitter_count_xml", "unique_submitter_count"],
            "T1 unique submitter count",
        ),
    ),
    (
        "aggregate_review_stars",
        resolve_column(
            t1_schema_columns,
            ["aggregate_review_stars", "review_stars"],
            "T1 aggregate review stars",
        ),
    ),
    (
        "aggregate_conflict_flag",
        resolve_column(
            t1_schema_columns,
            ["aggregate_conflict_flag", "conflict_flag"],
            "T1 aggregate conflict flag",
        ),
    ),
    (
        "scv_group_counts_json",
        resolve_column(
            t1_schema_columns,
            ["scv_group_counts_json", "group_counts_json"],
            "T1 SCV group counts",
        ),
    ),
    (
        "scv_count",
        resolve_column(
            t1_schema_columns,
            ["scv_count_xml", "scv_count"],
            "T1 SCV count",
        ),
    ),
])

read_columns = list(dict.fromkeys(t1_columns.values()))
t1 = pd.read_parquet(T1_PARQUET, columns=read_columns).copy()

rcv_t1 = (
    t1[t1_columns["rcv_accession"]]
    .astype("string")
    .str.strip()
    .str.upper()
)

gene_t1 = t1[t1_columns["target_gene"]].map(normalize_gene)

axis_t1 = (
    t1[t1_columns["classification_axis"]]
    .astype("string")
    .fillna("NoClassification")
    .str.strip()
    .replace("", "NoClassification")
)

cutoff_t1_timestamp = pd.to_datetime(
    t1[t1_columns["embedded_cutoff"]],
    errors="coerce",
    utc=True,
).dt.tz_convert(None)

cutoff_t1 = cutoff_t1_timestamp.dt.strftime("%Y-%m-%d")

last_evaluated_t1 = pd.to_datetime(
    t1[t1_columns["aggregate_last_evaluated"]],
    errors="coerce",
    utc=True,
).dt.tz_convert(None)

recency_days_t1 = (
    cutoff_t1_timestamp - last_evaluated_t1
).dt.days.astype(float)

recency_score_t1 = (
    1.0 - recency_days_t1 / MAXIMUM_OBSERVATION_WINDOW_DAYS
).clip(0.0, 1.0)
recency_score_t1.loc[last_evaluated_t1.isna()] = np.nan

recency_missing_flag_t1 = last_evaluated_t1.isna().astype(float)

submitter_count_t1 = pd.to_numeric(
    t1[t1_columns["unique_submitter_count"]],
    errors="coerce",
).astype(float)

log_submitter_t1 = np.log1p(submitter_count_t1)

submitter_diversity_score_t1 = (
    (log_submitter_t1 - SUBMITTER_LOG_MIN)
    / (SUBMITTER_LOG_MAX - SUBMITTER_LOG_MIN)
).clip(0.0, 1.0)

review_confidence_t1 = pd.to_numeric(
    t1[t1_columns["aggregate_review_stars"]],
    errors="coerce",
).astype(float)

conflict_t1 = t1[t1_columns["aggregate_conflict_flag"]].map(
    boolish_to_float
).astype(float)

entropy_results = t1[t1_columns["scv_group_counts_json"]].map(
    normalized_entropy_from_group_counts
)

entropy_t1 = pd.Series(
    [item[0] for item in entropy_results],
    index=t1.index,
    dtype=float,
)
group_count_sum_t1 = pd.Series(
    [item[1] for item in entropy_results],
    index=t1.index,
    dtype=float,
)
positive_group_count_t1 = pd.Series(
    [item[2] for item in entropy_results],
    index=t1.index,
    dtype="int64",
)
entropy_status_t1 = pd.Series(
    [item[3] for item in entropy_results],
    index=t1.index,
    dtype="string",
)

scv_count_t1 = pd.to_numeric(
    t1[t1_columns["scv_count"]],
    errors="coerce",
).astype(float)

scv_group_count_mismatch_mask = ~np.isclose(
    group_count_sum_t1.to_numpy(),
    scv_count_t1.to_numpy(),
    rtol=0.0,
    atol=0.0,
    equal_nan=False,
)
scv_group_count_mismatches = int(
    scv_group_count_mismatch_mask.sum()
)

post_cutoff_dates = int(
    (
        last_evaluated_t1.notna()
        & cutoff_t1_timestamp.notna()
        & (last_evaluated_t1 > cutoff_t1_timestamp)
    ).sum()
)

t1_features = pd.DataFrame({
    "recency_score": recency_score_t1,
    "recency_missing_flag": recency_missing_flag_t1,
    "submitter_diversity_score": submitter_diversity_score_t1,
    "review_confidence": review_confidence_t1,
    "aggregate_conflict_flag": conflict_t1,
    "scv_group_entropy_normalized": entropy_t1,
})

# Keep the reconstructed model-input frame in memory; only the authorized score table is persisted.


# --------------------------------------------------------------------------------------------------
# 11. VERIFY FROZEN MODEL PACKAGES AND PREPROCESSING APPLICABILITY BEFORE SCORING
# --------------------------------------------------------------------------------------------------

full_artifact = joblib.load(FULL_MODEL_PATH)
no_star_artifact = joblib.load(NO_STAR_MODEL_PATH)

model_jobs = OrderedDict([
    ("full_ges", (full_artifact, FULL_FEATURES)),
    ("no_star_ges", (no_star_artifact, NO_STAR_FEATURES)),
])

model_results = OrderedDict()
model_inventory_rows = []

for model_name, (artifact, expected_features) in model_jobs.items():
    pipeline, pipeline_location = extract_pipeline(artifact, model_name)

    imputer_name, imputer = get_exact_component(pipeline, SimpleImputer)
    scaler_name, scaler = get_exact_component(pipeline, StandardScaler)
    classifier_name, classifier = get_exact_component(
        pipeline, LogisticRegression
    )

    feature_order, feature_order_source, feature_order_verified = (
        find_feature_order(artifact, pipeline, expected_features)
    )

    fitted_state_verified = bool(
        hasattr(imputer, "statistics_")
        and hasattr(scaler, "mean_")
        and hasattr(scaler, "scale_")
        and hasattr(classifier, "coef_")
        and hasattr(classifier, "intercept_")
        and hasattr(classifier, "classes_")
    )

    settings_verified = model_settings_match(classifier)

    input_frame = t1_features[expected_features].copy()

    transformed = preprocess_without_scoring(
        pipeline,
        input_frame,
    )

    transformed_shape = np.asarray(
        transformed.toarray() if sparse.issparse(transformed) else transformed
    ).shape

    result = {
        "model": model_name,
        "pipeline_location": pipeline_location,
        "pipeline_steps": [name for name, _ in pipeline.steps],
        "imputer_step": imputer_name,
        "scaler_step": scaler_name,
        "classifier_step": classifier_name,
        "feature_order": feature_order,
        "feature_order_source": feature_order_source,
        "feature_order_verified": bool(feature_order_verified),
        "fitted_state_verified": fitted_state_verified,
        "settings_verified": settings_verified,
        "rows": int(transformed_shape[0]),
        "columns": int(transformed_shape[1]),
        "all_finite": transformed_values_are_finite(transformed),
        "classifier_scoring_applied": False,
    }

    model_results[model_name] = result

    model_inventory_rows.append({
        "model": model_name,
        "artifact_path": str(
            FULL_MODEL_PATH
            if model_name == "full_ges"
            else NO_STAR_MODEL_PATH
        ),
        "artifact_sha256": sha256_file(
            FULL_MODEL_PATH
            if model_name == "full_ges"
            else NO_STAR_MODEL_PATH
        ),
        "pipeline_location": pipeline_location,
        "pipeline_steps": json.dumps(result["pipeline_steps"]),
        "feature_order_source": feature_order_source,
        "expected_feature_order": json.dumps(expected_features),
        "observed_feature_order": json.dumps(feature_order),
        "feature_order_verified": feature_order_verified,
        "fitted_state_verified": fitted_state_verified,
        "settings_verified": settings_verified,
        "t1_preprocessing_rows": result["rows"],
        "t1_preprocessing_columns": result["columns"],
        "t1_preprocessing_all_finite": result["all_finite"],
        "classifier_scoring_applied": False,
    })

model_inventory = pd.DataFrame(model_inventory_rows)


# --------------------------------------------------------------------------------------------------
# 12. BUILD FEATURE AND CLASSIFICATION-AXIS INVENTORIES
# --------------------------------------------------------------------------------------------------

feature_inventory_rows = []

for feature in FULL_FEATURES:
    t0_values = pd.to_numeric(t0[feature], errors="coerce").astype(float)
    t1_values = pd.to_numeric(t1_features[feature], errors="coerce").astype(float)

    t0_nonmissing = t0_values.dropna()
    t1_nonmissing = t1_values.dropna()

    t0_min = float(t0_nonmissing.min()) if len(t0_nonmissing) else np.nan
    t0_max = float(t0_nonmissing.max()) if len(t0_nonmissing) else np.nan
    domain_upper = 4.0 if feature == "review_confidence" else 1.0

    feature_inventory_rows.append({
        "feature": feature,
        "used_by_full_ges": feature in FULL_FEATURES,
        "used_by_no_star_ges": feature in NO_STAR_FEATURES,
        "t0_rows": int(len(t0_values)),
        "t0_missing": int(t0_values.isna().sum()),
        "t0_min": t0_min,
        "t0_max": t0_max,
        "t1_rows": int(len(t1_values)),
        "t1_missing": int(t1_values.isna().sum()),
        "t1_min": (
            float(t1_nonmissing.min()) if len(t1_nonmissing) else np.nan
        ),
        "t1_max": (
            float(t1_nonmissing.max()) if len(t1_nonmissing) else np.nan
        ),
        "t1_below_t0_range": int(
            (t1_nonmissing < t0_min).sum()
        ) if np.isfinite(t0_min) else 0,
        "t1_above_t0_range": int(
            (t1_nonmissing > t0_max).sum()
        ) if np.isfinite(t0_max) else 0,
        "t1_outside_frozen_domain": int(
            (
                (t1_nonmissing < 0.0)
                | (t1_nonmissing > domain_upper)
            ).sum()
        ),
        "t1_infinite_values": int(
            np.isinf(t1_values.to_numpy()).sum()
        ),
        "frozen_median_imputer_available": True,
        "row_level_feature_persisted": False,
    })

feature_inventory = pd.DataFrame(feature_inventory_rows)

axis_inventory = (
    pd.DataFrame({
        "classification_axis": axis_t1,
        "target_gene": gene_t1,
    })
    .groupby(["classification_axis", "target_gene"], dropna=False)
    .size()
    .reset_index(name="rows")
)


def axis_policy(axis):
    if axis == "GermlineClassification":
        return "primary_germline_domain_candidate"
    if axis == "NoClassification":
        return "retain_as_evidence_only_not_as_classification"
    return "retain_separately_requires_prespecified_non_germline_policy"


axis_inventory["scientific_applicability"] = (
    axis_inventory["classification_axis"].map(axis_policy)
)
axis_inventory["included_in_preprocessing_check"] = True
axis_inventory["score_created_in_cell_7a2"] = False

observed_gene_counts = {
    str(key): int(value)
    for key, value in gene_t1.value_counts(dropna=False).items()
}

observed_axis_counts = {
    str(key): int(value)
    for key, value in axis_t1.value_counts(dropna=False).items()
}


# --------------------------------------------------------------------------------------------------
# 13. VERIFY THE FROZEN COMBINED-METADATA POLICY BEFORE APPLYING IT
# --------------------------------------------------------------------------------------------------

stage6a_policy_payload = json.loads(
    STAGE6A_POLICY.read_text(encoding="utf-8")
)
stage6a_policy_text = json.dumps(
    stage6a_policy_payload,
    sort_keys=True,
).lower()

combined_metadata_policy_identified = bool(
    "combined_metadata_instability_risk" in stage6a_policy_text
    and "frozen_before_outcome_label_load_or_temporal_performance"
        in stage6a_policy_text
)




# --------------------------------------------------------------------------------------------------
# 14. VERIFY FROZEN PREDICT_PROBA SEMANTICS ON T0
# --------------------------------------------------------------------------------------------------

def frozen_stable_probability(artifact, expected_features, frame, label):
    pipeline, pipeline_location = extract_pipeline(artifact, label)
    _, classifier = get_exact_component(pipeline, LogisticRegression)
    classes = np.asarray(classifier.classes_)
    stable_positions = np.flatnonzero(classes == 1)
    if len(stable_positions) != 1:
        raise AssertionError(
            f"{label} classifier classes do not contain exactly one stable class encoded as 1: {classes}"
        )
    probabilities = np.asarray(
        pipeline.predict_proba(frame[expected_features].copy()),
        dtype=float,
    )
    if probabilities.shape != (len(frame), len(classes)):
        raise AssertionError(f"Unexpected predict_proba shape for {label}: {probabilities.shape}")
    p_stable = probabilities[:, int(stable_positions[0])]
    if not np.isfinite(p_stable).all():
        raise AssertionError(f"Nonfinite stable probabilities generated by {label}.")
    if ((p_stable < 0.0) | (p_stable > 1.0)).any():
        raise AssertionError(f"Out-of-range stable probabilities generated by {label}.")
    return p_stable, pipeline_location, classes.tolist()

frozen_t0_scores = pd.read_parquet(
    STAGE4C_SCORE_TABLE,
    columns=[
        "t0_row_order",
        "rcv_accession",
        "full_ges_p_stable_t0",
        "no_star_ges_p_stable_t0",
    ],
).copy()

if len(frozen_t0_scores) != EXPECTED_STAGE4B_ROWS:
    raise AssertionError("Frozen Stage 4C T0 score-table row count is incorrect.")
if not np.array_equal(
    pd.to_numeric(frozen_t0_scores["t0_row_order"], errors="raise").to_numpy(dtype=np.int64),
    t0_row_order.to_numpy(dtype=np.int64),
):
    raise AssertionError("Frozen Stage 4C T0 score-table row order does not align with Stage 4B.")
if not np.array_equal(
    frozen_t0_scores["rcv_accession"].astype("string").str.strip().str.upper().to_numpy(dtype=str),
    t0_rcv.to_numpy(dtype=str),
):
    raise AssertionError("Frozen Stage 4C T0 score-table RCV order does not align with Stage 4B.")

full_t0_reproduced, full_pipeline_location, full_classes = frozen_stable_probability(
    full_artifact, FULL_FEATURES, t0, "full_ges"
)
no_star_t0_reproduced, no_star_pipeline_location, no_star_classes = frozen_stable_probability(
    no_star_artifact, NO_STAR_FEATURES, t0, "no_star_ges"
)

full_t0_frozen = pd.to_numeric(
    frozen_t0_scores["full_ges_p_stable_t0"], errors="raise"
).to_numpy(dtype=float)
no_star_t0_frozen = pd.to_numeric(
    frozen_t0_scores["no_star_ges_p_stable_t0"], errors="raise"
).to_numpy(dtype=float)

full_t0_probability_mismatches = int((~np.isclose(
    full_t0_reproduced,
    full_t0_frozen,
    rtol=1e-12,
    atol=1e-12,
    equal_nan=False,
)).sum())
no_star_t0_probability_mismatches = int((~np.isclose(
    no_star_t0_reproduced,
    no_star_t0_frozen,
    rtol=1e-12,
    atol=1e-12,
    equal_nan=False,
)).sum())

if full_t0_probability_mismatches or no_star_t0_probability_mismatches:
    raise AssertionError(
        "Frozen T0 predict_proba reproduction failed: "
        f"full={full_t0_probability_mismatches}, no_star={no_star_t0_probability_mismatches}."
    )


# --------------------------------------------------------------------------------------------------
# 15. MATERIALIZE THE AUTHORIZED T1 FULL-GES AND NO-STAR-GES SCORES IN MEMORY
# --------------------------------------------------------------------------------------------------

full_p_stable_t1, _, _ = frozen_stable_probability(
    full_artifact, FULL_FEATURES, t1_features, "full_ges"
)
no_star_p_stable_t1, _, _ = frozen_stable_probability(
    no_star_artifact, NO_STAR_FEATURES, t1_features, "no_star_ges"
)

full_instability_t1 = 1.0 - full_p_stable_t1
no_star_instability_t1 = 1.0 - no_star_p_stable_t1


# --------------------------------------------------------------------------------------------------
# 16. APPLY THE EXACT FROZEN STAGE 6A COMBINED-METADATA POLICY
# --------------------------------------------------------------------------------------------------

policy_status = stage6a_policy_payload.get("status")
policy_version = str(stage6a_policy_payload.get("version"))
policy_constants = stage6a_policy_payload.get("frozen_constants", {})
policy_definitions = stage6a_policy_payload.get("score_definitions", {})
combined_definition = policy_definitions.get("strong_combined_metadata_heuristic", {})
combined_weights = combined_definition.get("component_weights", {})
construction_requirements = stage6a_policy_payload.get("construction_requirements", {})

EXPECTED_POLICY_STATUS = "FROZEN_BEFORE_OUTCOME_LABEL_LOAD_OR_TEMPORAL_PERFORMANCE"
EXPECTED_COMBINED_COMPONENTS = [
    "review_stars_instability_risk",
    "conflict_instability_risk",
    "recency_instability_risk",
    "recency_missing_instability_component",
    "submitter_instability_risk",
    "entropy_instability_risk",
]

if policy_status != EXPECTED_POLICY_STATUS or policy_version != "1.0.0":
    raise AssertionError("Frozen Stage 6A comparator-policy identity/status verification failed.")
if not combined_metadata_policy_identified:
    raise AssertionError("Frozen combined-metadata policy was not identified by Cell 7A2 logic.")
if list(combined_definition.get("components", [])) != EXPECTED_COMBINED_COMPONENTS:
    raise AssertionError("Frozen combined-metadata component order does not match the expected policy.")
if set(combined_weights) != set(EXPECTED_COMBINED_COMPONENTS):
    raise AssertionError("Frozen combined-metadata component-weight keys are incorrect.")
if not np.isclose(sum(float(v) for v in combined_weights.values()), 1.0, rtol=0.0, atol=1e-15):
    raise AssertionError("Frozen combined-metadata weights do not sum to one.")
if any(bool(construction_requirements.get(key, True)) for key in [
    "allow_outcome_label_load",
    "allow_outcome_table_load",
    "allow_score_outcome_join",
    "allow_temporal_performance_calculation",
    "allow_threshold_optimization",
    "allow_weight_optimization",
]):
    raise AssertionError("Frozen policy unexpectedly permits a prohibited operation.")

review_scaling_max = float(policy_constants["review_star_scaling_maximum"])
recency_window_days = float(policy_constants["recency_maximum_observation_window_days"])
recency_missing_imputation = float(
    policy_constants["recency_median_instability_risk_for_missing_imputation"]
)
submitter_min = float(policy_constants["submitter_minimum_log1p_count"])
submitter_max = float(policy_constants["submitter_maximum_log1p_count"])

if not np.isclose(recency_window_days, MAXIMUM_OBSERVATION_WINDOW_DAYS, rtol=0.0, atol=1e-12):
    raise AssertionError("Stage 6A recency constant does not match the frozen Stage 4B transformation.")
if not np.isclose(submitter_min, SUBMITTER_LOG_MIN, rtol=0.0, atol=1e-15):
    raise AssertionError("Stage 6A submitter minimum does not match Stage 4B.")
if not np.isclose(submitter_max, SUBMITTER_LOG_MAX, rtol=0.0, atol=1e-15):
    raise AssertionError("Stage 6A submitter maximum does not match Stage 4B.")

review_stars_t1 = review_confidence_t1.astype(float)
review_risk_t1 = ((review_scaling_max - review_stars_t1) / review_scaling_max).clip(0.0, 1.0)
conflict_risk_t1 = conflict_t1.astype(float)
recency_risk_t1 = (recency_days_t1 / recency_window_days).clip(0.0, 1.0)
recency_risk_t1.loc[recency_missing_flag_t1.eq(1.0)] = recency_missing_imputation
recency_missing_component_t1 = recency_missing_flag_t1.astype(float)
submitter_risk_t1 = (1.0 - submitter_diversity_score_t1).clip(0.0, 1.0)
entropy_risk_t1 = entropy_t1.clip(0.0, 1.0)

component_frame = pd.DataFrame({
    "review_stars_instability_risk": review_risk_t1,
    "conflict_instability_risk": conflict_risk_t1,
    "recency_instability_risk": recency_risk_t1,
    "recency_missing_instability_component": recency_missing_component_t1,
    "submitter_instability_risk": submitter_risk_t1,
    "entropy_instability_risk": entropy_risk_t1,
})

combined_metadata_t1 = np.zeros(len(component_frame), dtype=float)
for component in EXPECTED_COMBINED_COMPONENTS:
    combined_metadata_t1 += (
        component_frame[component].to_numpy(dtype=float)
        * float(combined_weights[component])
    )
combined_metadata_t1 = np.clip(combined_metadata_t1, 0.0, 1.0)


# --------------------------------------------------------------------------------------------------
# 17. ASSEMBLE THE SINGLE AUTHORIZED ROW-LEVEL T1 SCORE TABLE
# --------------------------------------------------------------------------------------------------

score_table = pd.DataFrame({
    "t1_row_order": np.arange(len(t1), dtype=np.int64),
    "rcv_accession": rcv_t1.astype(str),
    "target_gene": gene_t1.astype(str),
    "classification_axis": axis_t1.astype(str),
    **{column: component_frame[column].astype(float) for column in EXPECTED_COMBINED_COMPONENTS},
    "combined_metadata_instability_risk": combined_metadata_t1.astype(float),
    "full_ges_p_stable_t1": full_p_stable_t1.astype(float),
    "full_ges_instability_risk_t1": full_instability_t1.astype(float),
    "no_star_ges_p_stable_t1": no_star_p_stable_t1.astype(float),
    "no_star_ges_instability_risk_t1": no_star_instability_t1.astype(float),
    "comparator_policy_version": policy_version,
    "comparator_policy_sha256": sha256_file(STAGE6A_POLICY),
    "cell_7a2_manifest_sha256": sha256_file(CELL_7A2_MANIFEST),
    "stage7a3_version": "1.0.0",
    "model_fitted_or_refitted": False,
    "threshold_or_weight_optimized": False,
    "score_rank_constructed": False,
    "hard_exclusion_applied": False,
    "outcome_labels_loaded": False,
    "temporal_performance_evaluated": False,
    "rag_corpus_constructed": False,
    "embeddings_constructed": False,
    "llm_called": False,
})

SCORE_COLUMNS = [
    "review_stars_instability_risk",
    "conflict_instability_risk",
    "recency_instability_risk",
    "recency_missing_instability_component",
    "submitter_instability_risk",
    "entropy_instability_risk",
    "combined_metadata_instability_risk",
    "full_ges_p_stable_t1",
    "full_ges_instability_risk_t1",
    "no_star_ges_p_stable_t1",
    "no_star_ges_instability_risk_t1",
]

PROHIBITED_COLUMN_PATTERNS = [
    r"(^|_)rank($|_)",
    r"threshold",
    r"predicted_.*0_5",
    r"future_instability_outcome",
    r"primary_outcome",
    r"embedding",
    r"retrieval",
    r"prompt",
    r"llm_output",
]

# These columns are explicit FALSE-valued governance/audit assertions, not scientific
# payloads. They must remain in the frozen table so downstream readers can verify that
# the prohibited operations did not occur. Excluding their names from the pattern scan
# does not authorize those operations; their FALSE values are independently QC-checked.
PROHIBITION_AUDIT_FLAG_COLUMNS = {
    "model_fitted_or_refitted",
    "threshold_or_weight_optimized",
    "score_rank_constructed",
    "hard_exclusion_applied",
    "outcome_labels_loaded",
    "temporal_performance_evaluated",
    "rag_corpus_constructed",
    "embeddings_constructed",
    "llm_called",
}

prohibited_column_matches = sorted({
    (column, pattern)
    for column in score_table.columns
    if column not in PROHIBITION_AUDIT_FLAG_COLUMNS
    for pattern in PROHIBITED_COLUMN_PATTERNS
    if re.search(pattern, column, flags=re.I)
})


# --------------------------------------------------------------------------------------------------
# 18. PRE-WRITE QUALITY-CONTROL REGISTER — FAIL BEFORE OUTPUT
# --------------------------------------------------------------------------------------------------

immutable_hashes_after_scoring = {
    key: sha256_file(path)
    for key, path in upstream_paths.items()
}

score_qc_rows = []
for column in SCORE_COLUMNS:
    values = pd.to_numeric(score_table[column], errors="coerce").to_numpy(dtype=float)
    score_qc_rows.append({
        "score_column": column,
        "rows": int(len(values)),
        "missing_values": int(np.isnan(values).sum()),
        "infinite_values": int(np.isinf(values).sum()),
        "minimum": float(np.nanmin(values)),
        "maximum": float(np.nanmax(values)),
        "outside_unit_interval": int(((values < 0.0) | (values > 1.0)).sum()),
        "nonconstant": bool(float(np.nanstd(values)) > 0.0),
    })
score_qc_inventory = pd.DataFrame(score_qc_rows)

schema_inventory = pd.DataFrame([
    {
        "column_order": index,
        "column_name": column,
        "dtype": str(score_table[column].dtype),
        "role": (
            "identifier_or_stratum"
            if column in {"t1_row_order", "rcv_accession", "target_gene", "classification_axis"}
            else "authorized_score_or_component"
            if column in SCORE_COLUMNS
            else "frozen_provenance_or_prohibition_flag"
        ),
    }
    for index, column in enumerate(score_table.columns)
])

qc_checks = OrderedDict([
    ("cell_7a2_manifest_exact_hash_verified", sha256_file(CELL_7A2_MANIFEST) == EXPECTED_CELL_7A2_MANIFEST_SHA256),
    ("cell_7a2_terminal_decision_verified", cell_7a2_payload.get("terminal_decision") == EXPECTED_CELL_7A2_DECISION),
    ("cell_7a2_62_of_62_qc_verified", int(qc_summary_7a2.get("passed_checks", -1)) == 62 and int(qc_summary_7a2.get("failed_checks", -1)) == 0),
    ("all_frozen_input_hashes_verified", all(observed_hashes[key] == EXPECTED_HASHES[key] for key in EXPECTED_HASHES)),
    ("all_cell_7a2_output_hashes_verified", len(cell_7a2_output_paths) == len(cell_7a2_payload.get("output_artifacts", {}))),
    ("upstream_immutability_after_scoring", immutable_hashes_before == immutable_hashes_after_scoring),
    ("t1_row_count_verified", len(score_table) == EXPECTED_T1_ROWS),
    ("t1_row_order_zero_based", np.array_equal(score_table["t1_row_order"].to_numpy(), np.arange(EXPECTED_T1_ROWS, dtype=np.int64))),
    ("t1_rcv_complete", score_table["rcv_accession"].notna().all() and ~score_table["rcv_accession"].eq("").any()),
    ("t1_rcv_unique", score_table["rcv_accession"].nunique(dropna=False) == EXPECTED_T1_ROWS),
    ("t1_gene_counts_verified", score_table["target_gene"].value_counts().to_dict() == EXPECTED_T1_GENE_COUNTS),
    ("t1_axis_counts_verified", score_table["classification_axis"].value_counts().to_dict() == EXPECTED_T1_AXIS_COUNTS),
    ("full_t0_predict_proba_exactly_reproduced", full_t0_probability_mismatches == 0),
    ("no_star_t0_predict_proba_exactly_reproduced", no_star_t0_probability_mismatches == 0),
    ("full_classifier_classes_verified", full_classes == [0, 1]),
    ("no_star_classifier_classes_verified", no_star_classes == [0, 1]),
    ("full_pipeline_location_recorded", bool(full_pipeline_location)),
    ("no_star_pipeline_location_recorded", bool(no_star_pipeline_location)),
    ("combined_policy_status_verified", policy_status == EXPECTED_POLICY_STATUS),
    ("combined_policy_version_verified", policy_version == "1.0.0"),
    ("combined_policy_weights_sum_to_one", np.isclose(sum(float(v) for v in combined_weights.values()), 1.0, rtol=0.0, atol=1e-15)),
    ("combined_policy_has_six_components", len(combined_weights) == 6),
    ("all_score_columns_present", all(column in score_table.columns for column in SCORE_COLUMNS)),
    ("all_scores_complete", score_qc_inventory["missing_values"].eq(0).all()),
    ("all_scores_finite", score_qc_inventory["infinite_values"].eq(0).all()),
    ("all_scores_in_unit_interval", score_qc_inventory["outside_unit_interval"].eq(0).all()),
    ("all_scores_nonconstant", score_qc_inventory["nonconstant"].eq(True).all()),
    ("full_probability_risk_complements", np.allclose(score_table["full_ges_p_stable_t1"] + score_table["full_ges_instability_risk_t1"], 1.0, rtol=0.0, atol=1e-15)),
    ("no_star_probability_risk_complements", np.allclose(score_table["no_star_ges_p_stable_t1"] + score_table["no_star_ges_instability_risk_t1"], 1.0, rtol=0.0, atol=1e-15)),
    ("combined_score_exact_weighted_sum", np.allclose(score_table["combined_metadata_instability_risk"], sum(score_table[c] * float(combined_weights[c]) for c in EXPECTED_COMBINED_COMPONENTS), rtol=1e-15, atol=1e-15)),
    ("policy_hash_on_every_row", score_table["comparator_policy_sha256"].eq(sha256_file(STAGE6A_POLICY)).all()),
    ("cell_7a2_hash_on_every_row", score_table["cell_7a2_manifest_sha256"].eq(EXPECTED_CELL_7A2_MANIFEST_SHA256).all()),
    ("model_fitting_flag_false", score_table["model_fitted_or_refitted"].eq(False).all()),
    ("optimization_flag_false", score_table["threshold_or_weight_optimized"].eq(False).all()),
    ("rank_flag_false", score_table["score_rank_constructed"].eq(False).all()),
    ("hard_exclusion_flag_false", score_table["hard_exclusion_applied"].eq(False).all()),
    ("outcome_labels_flag_false", score_table["outcome_labels_loaded"].eq(False).all()),
    ("temporal_performance_flag_false", score_table["temporal_performance_evaluated"].eq(False).all()),
    ("rag_corpus_flag_false", score_table["rag_corpus_constructed"].eq(False).all()),
    ("embeddings_flag_false", score_table["embeddings_constructed"].eq(False).all()),
    ("llm_flag_false", score_table["llm_called"].eq(False).all()),
    ("no_prohibited_columns_materialized", len(prohibited_column_matches) == 0),
])

failed_checks = [name for name, passed in qc_checks.items() if not bool(passed)]
passed_checks = len(qc_checks) - len(failed_checks)
all_checks_passed = len(failed_checks) == 0

if not all_checks_passed:
    raise RuntimeError(
        "Cell 7A3 failed before output. Failed checks:\n- " + "\n- ".join(failed_checks)
    )


# --------------------------------------------------------------------------------------------------
# 19. IDEMPOTENT, CHECKSUM-PROTECTED OUTPUT WRITERS
# --------------------------------------------------------------------------------------------------

import pyarrow as pa


def stable_write_parquet(path: Path, frame: pd.DataFrame) -> str:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}")
    table = pa.Table.from_pandas(frame, preserve_index=False).replace_schema_metadata(None)
    pq.write_table(
        table,
        temporary,
        compression="zstd",
        use_dictionary=True,
        write_statistics=True,
        version="2.6",
        data_page_version="1.0",
    )
    proposed_hash = sha256_file(temporary)
    if path.exists():
        existing_hash = sha256_file(path)
        if existing_hash != proposed_hash:
            temporary.unlink(missing_ok=True)
            raise RuntimeError(
                "Refusing to overwrite a nonidentical frozen Cell 7A3 Parquet artifact.\n"
                f"Path: {path}\nExisting SHA-256: {existing_hash}\nProposed SHA-256: {proposed_hash}"
            )
        temporary.unlink(missing_ok=True)
    else:
        os.replace(temporary, path)
    return sha256_file(path)

existing_created_utc = None
for candidate in [OUTPUTS["manifest"], OUTPUTS["qc"], OUTPUTS["materialization_report"]]:
    if candidate.exists():
        try:
            existing_created_utc = json.loads(candidate.read_text(encoding="utf-8")).get("created_utc")
            if existing_created_utc:
                break
        except Exception:
            pass
created_utc = existing_created_utc or datetime.now(timezone.utc).isoformat()

score_table_hash = stable_write_parquet(OUTPUTS["score_table"], score_table)
write_sidecar(OUTPUTS["score_table"])
schema_hash = stable_write_csv(OUTPUTS["schema_inventory"], schema_inventory)
write_sidecar(OUTPUTS["schema_inventory"])
score_qc_inventory_hash = stable_write_csv(OUTPUTS["score_qc_inventory"], score_qc_inventory)
write_sidecar(OUTPUTS["score_qc_inventory"])

# Fresh readback of the row-level frozen score table.
readback = pd.read_parquet(OUTPUTS["score_table"])
readback_checks = OrderedDict([
    ("readback_rows", len(readback) == EXPECTED_T1_ROWS),
    ("readback_columns", list(readback.columns) == list(score_table.columns)),
    ("readback_row_order", np.array_equal(readback["t1_row_order"].to_numpy(dtype=np.int64), score_table["t1_row_order"].to_numpy(dtype=np.int64))),
    ("readback_rcv_order", np.array_equal(readback["rcv_accession"].astype(str).to_numpy(), score_table["rcv_accession"].astype(str).to_numpy())),
    ("readback_score_values", all(np.allclose(pd.to_numeric(readback[c]).to_numpy(dtype=float), pd.to_numeric(score_table[c]).to_numpy(dtype=float), rtol=0.0, atol=0.0) for c in SCORE_COLUMNS)),
    ("score_table_sidecar", sidecar_is_valid(OUTPUTS["score_table"])),
    ("schema_sidecar", sidecar_is_valid(OUTPUTS["schema_inventory"])),
    ("score_qc_inventory_sidecar", sidecar_is_valid(OUTPUTS["score_qc_inventory"])),
])
if not all(readback_checks.values()):
    raise RuntimeError(
        "Cell 7A3 output readback failed: "
        + ", ".join(name for name, passed in readback_checks.items() if not passed)
    )

terminal_decision = (
    "PASS_STAGE7A3_FROZEN_T1_FULL_GES_NO_STAR_GES_AND_COMBINED_METADATA_"
    "SCORES_MATERIALIZED_CHECKSUM_PROTECTED_T0_PREDICT_PROBA_REPRODUCED_"
    "NO_FITTING_NO_THRESHOLDING_NO_RANKING_NO_RAG_CORPUS_EMBEDDINGS_OR_LLM_"
    "NEXT_STAGE_REQUIRES_SEPARATE_PROTOCOL_FREEZE"
)

materialization_report = {
    "cell_id": CELL_ID,
    "notebook_name": NOTEBOOK_NAME,
    "package_version": PACKAGE_VERSION,
    "created_utc": created_utc,
    "purpose": "Frozen T1 Full-GES, No-star-GES, and combined-metadata score materialization.",
    "authorization": {
        "cell_7a2_manifest_path": str(CELL_7A2_MANIFEST),
        "cell_7a2_manifest_sha256": sha256_file(CELL_7A2_MANIFEST),
        "cell_7a2_terminal_decision_verified": True,
        "cell_7a2_qc_passed": "62/62",
    },
    "upstream_hashes": {key: sha256_file(path) for key, path in upstream_paths.items()},
    "t0_predict_proba_reproduction": {
        "full_ges_mismatches": full_t0_probability_mismatches,
        "no_star_ges_mismatches": no_star_t0_probability_mismatches,
        "full_classifier_classes": full_classes,
        "no_star_classifier_classes": no_star_classes,
    },
    "t1_score_table": {
        "path": str(OUTPUTS["score_table"]),
        "sha256": score_table_hash,
        "rows": len(score_table),
        "columns": len(score_table.columns),
        "score_columns": SCORE_COLUMNS,
    },
    "combined_metadata_policy": {
        "path": str(STAGE6A_POLICY),
        "sha256": sha256_file(STAGE6A_POLICY),
        "version": policy_version,
        "status": policy_status,
        "components": EXPECTED_COMBINED_COMPONENTS,
        "weights": {k: float(v) for k, v in combined_weights.items()},
    },
    "scientific_boundary": {
        "model_fitted_or_refitted": False,
        "model_recalibrated_or_tuned": False,
        "threshold_or_weight_optimized": False,
        "rank_constructed": False,
        "hard_exclusion_applied": False,
        "outcome_table_loaded": False,
        "temporal_performance_evaluated": False,
        "evidence_packet_materialized": False,
        "rag_corpus_constructed": False,
        "embeddings_constructed": False,
        "retrieval_executed": False,
        "question_set_constructed": False,
        "llm_called": False,
    },
    "software": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "pyarrow": __import__("pyarrow").__version__,
        "scikit_learn": __import__("sklearn").__version__,
        "joblib": joblib.__version__,
    },
}

stable_write_json(OUTPUTS["materialization_report"], materialization_report)
write_sidecar(OUTPUTS["materialization_report"])

qc_payload = {
    "cell_id": CELL_ID,
    "notebook_name": NOTEBOOK_NAME,
    "package_version": PACKAGE_VERSION,
    "created_utc": created_utc,
    "total_checks": len(qc_checks) + len(readback_checks),
    "passed_checks": passed_checks + sum(bool(v) for v in readback_checks.values()),
    "failed_checks": [],
    "all_checks_passed": True,
    "prewrite_checks": [{"check": k, "passed": bool(v)} for k, v in qc_checks.items()],
    "readback_checks": [{"check": k, "passed": bool(v)} for k, v in readback_checks.items()],
    "prohibited_column_scan": {
        "excluded_false_governance_flags": sorted(PROHIBITION_AUDIT_FLAG_COLUMNS),
        "matches": [
            {"column": column, "pattern": pattern}
            for column, pattern in prohibited_column_matches
        ],
    },
    "decision": terminal_decision,
}
stable_write_json(OUTPUTS["qc"], qc_payload)
write_sidecar(OUTPUTS["qc"])

output_records = {}
for key, path in OUTPUTS.items():
    if key == "manifest":
        continue
    if not sidecar_is_valid(path):
        raise AssertionError(f"Fresh Cell 7A3 output sidecar failed: {path}")
    output_records[key] = {
        "path": str(path),
        "sha256": sha256_file(path),
        "sidecar_path": str(sidecar_path(path)),
        "sidecar_sha256": sha256_file(sidecar_path(path)),
    }

manifest_payload = {
    "cell_id": CELL_ID,
    "notebook_name": NOTEBOOK_NAME,
    "package_version": PACKAGE_VERSION,
    "created_utc": created_utc,
    "purpose": "Freeze the authorized T1 Full-GES, No-star-GES, and combined-metadata scores.",
    "upstream_lineage": {key: {"path": str(path), "sha256": sha256_file(path)} for key, path in upstream_paths.items()},
    "output_artifacts": output_records,
    "qc": {
        "path": str(OUTPUTS["qc"]),
        "sha256": sha256_file(OUTPUTS["qc"]),
        "passed_checks": int(qc_payload["passed_checks"]),
        "failed_checks": 0,
        "total_checks": int(qc_payload["total_checks"]),
    },
    "scientific_boundary": materialization_report["scientific_boundary"],
    "decision": terminal_decision,
    "terminal_decision": terminal_decision,
    "next_authorized_cell": None,
    "next_required_action": (
        "Freeze a separate downstream protocol/cell authorization before any evidence-packet, "
        "RAG-corpus, embedding, retrieval, question-set, prompt, or LLM operation."
    ),
}
stable_write_json(OUTPUTS["manifest"], manifest_payload)
write_sidecar(OUTPUTS["manifest"])

if not sidecar_is_valid(OUTPUTS["manifest"]):
    raise AssertionError("Cell 7A3 manifest sidecar verification failed.")
manifest_readback = json.loads(OUTPUTS["manifest"].read_text(encoding="utf-8"))
if manifest_readback.get("terminal_decision") != terminal_decision:
    raise AssertionError("Cell 7A3 manifest terminal-decision readback failed.")
if manifest_readback.get("next_authorized_cell") is not None:
    raise AssertionError("Cell 7A3 must not auto-authorize a later RAG cell.")

# Final immutability recheck after every output has been written.
immutable_hashes_final = {key: sha256_file(path) for key, path in upstream_paths.items()}
if immutable_hashes_final != immutable_hashes_before:
    raise AssertionError("One or more frozen upstream artifacts changed during Cell 7A3.")


# --------------------------------------------------------------------------------------------------
# 20. CONTROLLED FINAL OUTPUT
# --------------------------------------------------------------------------------------------------

separator = "=" * 144
print("\n" + separator)
print("EXPERIMENT 2 — STAGE 7A — CELL 7A3")
print("FROZEN T1 FULL-GES, NO-STAR-GES, AND COMBINED-METADATA SCORE MATERIALIZATION")
print(separator)
print(f"Notebook                                      : {NOTEBOOK_NAME}")
print(f"Project root                                  : {ROOT}")

print("\nUPSTREAM AUTHORIZATION")
print(f"Cell 7A2 manifest SHA-256                     : {sha256_file(CELL_7A2_MANIFEST)}")
print("Cell 7A2 terminal PASS verified               : YES")
print("Cell 7A2 QC                                   : 62/62 PASS")
print("Model fitting authorized                      : NO")
print("Frozen T1 score materialization authorized    : YES")
print("Thresholding or ranking authorized            : NO")
print("RAG corpus / embeddings / LLM authorized      : NO")

print("\nFROZEN INPUT VERIFICATION")
print(f"Frozen source artifacts verified              : {len(observed_hashes)}/{len(EXPECTED_HASHES)}")
print(f"Cell 7A2 output artifacts reverified           : {len(cell_7a2_output_paths)}/{len(cell_7a2_output_paths)}")
print(f"T1 Parquet SHA-256                            : {sha256_file(T1_PARQUET)}")
print(f"Stage 4C Full-GES model SHA-256                : {sha256_file(FULL_MODEL_PATH)}")
print(f"Stage 4C No-star model SHA-256                 : {sha256_file(NO_STAR_MODEL_PATH)}")
print(f"Stage 6A comparator policy SHA-256             : {sha256_file(STAGE6A_POLICY)}")

print("\nT0 PREDICT_PROBA REPRODUCTION")
print(f"Full-GES probability mismatches                : {full_t0_probability_mismatches:,}")
print(f"No-star-GES probability mismatches             : {no_star_t0_probability_mismatches:,}")
print(f"Full-GES classifier classes                    : {full_classes}")
print(f"No-star-GES classifier classes                 : {no_star_classes}")

print("\nT1 SCORE MATERIALIZATION")
print(f"Rows                                           : {len(score_table):,}")
print(f"Unique RCV accessions                         : {score_table['rcv_accession'].nunique():,}")
print(f"Columns                                        : {len(score_table.columns):,}")
for row in score_qc_inventory.to_dict("records"):
    print(
        f"{row['score_column']:<46}: "
        f"min={row['minimum']:.12g} | max={row['maximum']:.12g} | "
        f"missing={int(row['missing_values'])} | outside[0,1]={int(row['outside_unit_interval'])}"
    )

print("\nCELL 7A3 FROZEN OUTPUTS")
for label, key in [
    ("T1 frozen score table", "score_table"),
    ("Score schema inventory", "schema_inventory"),
    ("Score range/QC inventory", "score_qc_inventory"),
    ("Materialization report", "materialization_report"),
    ("QC record", "qc"),
    ("Manifest", "manifest"),
]:
    path = OUTPUTS[key]
    print(f"{label:<46}: {path}")
    print(f"{'SHA-256':<46}: {sha256_file(path)}")

print(f"\nQC checks                                      : {qc_payload['passed_checks']}/{qc_payload['total_checks']} PASS")

print("\nSCIENTIFIC OPERATIONS")
print("Full-GES fitted or refitted                    : NO")
print("No-star GES fitted or refitted                 : NO")
print("Full-GES T1 score generated                    : YES")
print("No-star GES T1 score generated                 : YES")
print("Combined-metadata T1 score generated           : YES")
print("Threshold or weight optimization               : NO")
print("Score ranking constructed                      : NO")
print("Hard evidence exclusion applied                : NO")
print("Outcome labels loaded                          : NO")
print("Temporal performance evaluated                 : NO")
print("RAG corpus constructed                         : NO")
print("Embeddings constructed                         : NO")
print("LLM called                                     : NO")

print("\nNEXT AUTHORIZATION BOUNDARY")
print("Later RAG cell                                 : NOT AUTOMATICALLY AUTHORIZED")
print("Required next action                           : Freeze a separate downstream protocol")
print("                                                 before evidence packets, corpus, embeddings,")
print("                                                 retrieval, questions, prompts, or LLM calls")

print(f"\nFINAL DECISION                                : {terminal_decision}")
print(separator)


Mounted at /content/drive

EXPERIMENT 2 — STAGE 7A — CELL 7A3
FROZEN T1 FULL-GES, NO-STAR-GES, AND COMBINED-METADATA SCORE MATERIALIZATION
Notebook                                      : 05_GES_Aware_Genomic_RAG_Cell_7A3_V2.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study

UPSTREAM AUTHORIZATION
Cell 7A2 manifest SHA-256                     : 9c1b93467778e4ef580474dc85940a7dc2e5b840e0d46f889e2d3e19cbe46c34
Cell 7A2 terminal PASS verified               : YES
Cell 7A2 QC                                   : 62/62 PASS
Model fitting authorized                      : NO
Frozen T1 score materialization authorized    : YES
Thresholding or ranking authorized            : NO
RAG corpus / embeddings / LLM authorized      : NO

FROZEN INPUT VERIFICATION
Frozen source artifacts verified              : 19/19
Cell 7A2 output artifacts reverified           : 7/7
T1 Parquet SHA-256                            : 5713a11bdbf4804758cc011f9b2f302afc91fa1f8


## Required stopping rule

Accept Cell 7A3 only when the final code cell reports:

- every QC check passing,
- zero T0 Full-GES and No-star-GES probability mismatches,
- all 100,920 T1 rows materialized with complete finite scores in `[0,1]`,
- the long `PASS_STAGE7A3_...` terminal decision, and
- no later RAG cell automatically authorized.

Do not begin evidence-packet, corpus, embedding, retrieval, question, prompt, or LLM work from a failed or incomplete run.
